# Lab 11: Sorting Analysis

In this notebook you'll visualize how sorting algorithms work, measure their performance, and discover why Big-O isn't the whole story.

**Before you start:** Paste your completed sort functions into the cell below.

In [ ]:
# ── Paste your completed functions here ──────────────────────────
# You need all five: bubble_sort, short_bubble_sort, insertion_sort,
# bubble_sort_counted, and insertion_sort_counted




In [ ]:
# Quick sanity check — run this to make sure your functions work
assert bubble_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "bubble_sort failed"
assert short_bubble_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "short_bubble_sort failed"
assert insertion_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "insertion_sort failed"

r1, c1, e1 = bubble_sort_counted([3, 1, 2])
assert r1 == [1, 2, 3], "bubble_sort_counted failed"

r2, c2, d2 = insertion_sort_counted([3, 1, 2])
assert r2 == [1, 2, 3], "insertion_sort_counted failed"

print("All functions working!")

---
## Experiment 1: Watching the Mechanism

Before we measure performance, let's *see* how each algorithm sorts. The helper functions below print the list after every pass so you can watch the progress.

In [ ]:
def bubble_sort_visual(a_list):
    """Bubble sort with a printout after each pass."""
    a_list = a_list[:]  # work on a copy
    n = len(a_list)
    print(f"Start:  {a_list}")
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if a_list[j] > a_list[j + 1]:
                a_list[j], a_list[j + 1] = a_list[j + 1], a_list[j]
        # Show sorted vs unsorted regions
        sorted_start = n - 1 - i
        unsorted = a_list[:sorted_start]
        sorted_part = a_list[sorted_start:]
        print(f"Pass {i+1}: {unsorted} | {sorted_part}")
    return a_list


def insertion_sort_visual(a_list):
    """Insertion sort with a printout after each pass."""
    a_list = a_list[:]  # work on a copy
    n = len(a_list)
    print(f"Start:  {a_list}")
    for i in range(1, n):
        current_value = a_list[i]
        position = i - 1
        while position >= 0 and a_list[position] > current_value:
            a_list[position + 1] = a_list[position]
            position -= 1
        a_list[position + 1] = current_value
        # Show sorted vs unsorted regions
        sorted_part = a_list[:i + 1]
        unsorted = a_list[i + 1:]
        print(f"Pass {i}: {sorted_part} | {unsorted}")
    return a_list

In [ ]:
test_list = [54, 26, 93, 17, 77, 31, 44, 55, 20]

print("=" * 50)
print("BUBBLE SORT")
print("=" * 50)
bubble_sort_visual(test_list)

print()
print("=" * 50)
print("INSERTION SORT")
print("=" * 50)
insertion_sort_visual(test_list)

### Experiment 1 Questions

Study the output above and answer these questions:

**Q1:** In bubble sort, where do the sorted items accumulate — the left end or the right end of the list? In insertion sort, where do they accumulate?

*Your answer:*


**Q2:** Watch bubble sort's first few passes. How much does the unsorted region shrink after each pass? Now watch insertion sort — how much does the sorted region grow after each pass?

*Your answer:*


**Q3:** Which algorithm's progress is easier to follow visually? Why do you think that is?

*Your answer:*


---
## Experiment 2: The Comparison Race

Now let's measure. We'll run both counted sort functions on randomly generated lists of increasing size and plot comparisons and data moves separately.

In [ ]:
import random
import matplotlib.pyplot as plt

sizes = [100, 500, 1000, 5000]
bubble_comps = []
bubble_moves = []
insertion_comps = []
insertion_moves = []

for n in sizes:
    data = list(range(n))
    random.shuffle(data)

    _, bc, be = bubble_sort_counted(data[:])
    _, ic, im = insertion_sort_counted(data[:])

    bubble_comps.append(bc)
    bubble_moves.append(be)
    insertion_comps.append(ic)
    insertion_moves.append(im)

    print(f"n={n:>5}: Bubble comps={bc:>10,}  exchanges={be:>10,}  |  Insertion comps={ic:>10,}  data_moves={im:>10,}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Comparisons
ax1.plot(sizes, bubble_comps, 'o-', label='Bubble Sort', color='#e74c3c')
ax1.plot(sizes, insertion_comps, 's-', label='Insertion Sort', color='#2ecc71')
ax1.set_xlabel('List Size (n)')
ax1.set_ylabel('Comparisons')
ax1.set_title('Comparisons: Bubble vs Insertion')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Data moves
ax2.plot(sizes, bubble_moves, 'o-', label='Bubble Sort (exchanges)', color='#e74c3c')
ax2.plot(sizes, insertion_moves, 's-', label='Insertion Sort (shifts + placements)', color='#2ecc71')
ax2.set_xlabel('List Size (n)')
ax2.set_ylabel('Data Moves')
ax2.set_title('Data Movement: Bubble vs Insertion')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Experiment 2 Questions

**Q1:** Look at the comparisons graph. Do bubble sort and insertion sort make roughly the same number of comparisons, or is one consistently higher? Does this match what you'd expect from their shared O(n^2) classification?

*Your answer:*


**Q2:** Now look at the data moves graph. What do you see? Which algorithm moves data more efficiently?

*Your answer:*


**Q3:** If comparisons are roughly equal but data movement is very different, what does that tell you about using Big-O alone to predict real-world performance?

*Your answer:*


---
## Experiment 3: Best Case, Worst Case

Not all inputs are created equal. Let's see how each algorithm handles the best case (already sorted), worst case (reverse sorted), and average case (random).

In [ ]:
n = 1000

already_sorted = list(range(n))
reverse_sorted = list(range(n - 1, -1, -1))
random_list = list(range(n))
random.shuffle(random_list)

cases = {
    "Already sorted": already_sorted,
    "Reverse sorted": reverse_sorted,
    "Random": random_list,
}

print(f"{'Case':<20} {'Algorithm':<15} {'Comparisons':>12} {'Data Moves':>12}")
print("-" * 62)

for case_name, data in cases.items():
    _, bc, be = bubble_sort_counted(data[:])
    _, ic, im = insertion_sort_counted(data[:])
    print(f"{case_name:<20} {'Bubble':<15} {bc:>12,} {be:>12,}")
    print(f"{'':<20} {'Insertion':<15} {ic:>12,} {im:>12,}")
    print()

### Experiment 3 Questions

**Q1:** Which sort benefits the most from already-sorted input? Look at both comparisons and data moves — does either drop dramatically?

*Your answer:*


**Q2:** On reverse-sorted input, which sort performs worst? Why does that specific input cause the most work for that algorithm?

*Your answer:*


**Q3:** If you knew your data was *almost* sorted — say, one or two items out of place — which algorithm would you choose: bubble sort, short bubble sort, or insertion sort? Why? Think about your short bubble implementation — how would it handle nearly-sorted input?

*Your answer:*
